In [ ]:
from pathlib import Path
import sys

# Add the project root directory to sys.path
sys.path.append(str(Path().resolve().parent))
from src.constants.paths import SECRET_PATH


from src.processing.pde_ple import PDE, PLE, es
from src.processing.document import Document

# get all paths
from src.constants.constants import (
    USEFUL_EXTENSIONS,
    ENVIRONNEMENT_PATH,
    S3_RAW_DOCS_PATH,
    SERVICES_PATH,
    ENVIRONNEMENT_RAW_PATH,
)
import pandas as pd
from src.processing.utils import *

from elasticsearch import Elasticsearch, helpers
from langchain.text_splitter import SentenceTransformersTokenTextSplitter

model_name = "intfloat/multilingual-e5-large"
tokens_per_chunk = 512
chunk_overlap = 50

# Instancier le splitter
text_splitter = SentenceTransformersTokenTextSplitter(
    model_name=model_name,
    tokens_per_chunk=tokens_per_chunk,
    chunk_overlap=chunk_overlap,
)
# Configuration
indexes = ["uc202-rex","uc202-rex-cameleon"]
chunk_index = "uc202-rex-chunks"
chunk_size = 512
chunk_overlap = 128


for original_index in indexes:
# Scroll all documents from the original index
    scroll = "2m"
    query = {"query": {"match_all": {}}}
    response = es.search(index=original_index, scroll=scroll, size=1000, body=query)

    scroll_id = response["_scroll_id"]
    hits = response["hits"]["hits"]

    actions = []
    if not es.indices.exists(index=chunk_index):
        es.indices.create(index=chunk_index)

    while hits:
        for doc in hits:
            doc_id = doc["_id"]
            source = doc["_source"]
            content = source.get("content", "")

            # Skip empty content
            if not content.strip():
                continue

            # Chunk the content
            chunks = text_splitter.split_text(content)

            for idx, chunk in enumerate(chunks):
                chunk_doc = {
                    "_index": chunk_index,
                    "_source": {
                        "chunk_content": chunk,
                        "original_doc_id": doc_id,
                        "chunk_id": idx,
                        **{
                            k: v for k, v in source.items() if k != "content"
                        },  # Copy metadata except original content
                    },
                }
                actions.append(chunk_doc)

                # Bulk every 5000 chunks to avoid memory overload
                if len(actions) >= 5000:
                    helpers.bulk(es, actions)
                    actions = []

        # Scroll to next batch
        response = es.scroll(scroll_id=scroll_id, scroll=scroll)
        scroll_id = response["_scroll_id"]
        hits = response["hits"]["hits"]

    # Final bulk if any remaining
    if actions:
        helpers.bulk(es, actions)


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/elasticsearch/_sync/client/__init__.py:397: SecurityWarning: Connecting to 'https://noeyyalp.noe.edf.fr:29203' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [1]:
from pathlib import Path
import sys

# Add the project root directory to sys.path
sys.path.append(str(Path().resolve().parent))
from src.constants.paths import SECRET_PATH


from src.processing.pde_ple import PDE, PLE, es
from src.processing.document import Document

# get all paths
from src.constants.constants import (
    USEFUL_EXTENSIONS,
    ENVIRONNEMENT_PATH,
    S3_RAW_DOCS_PATH,
    SERVICES_PATH,
    ENVIRONNEMENT_RAW_PATH,
)
import pandas as pd
from src.processing.utils import *

from elasticsearch import Elasticsearch, helpers
from langchain.text_splitter import SentenceTransformersTokenTextSplitter


# Configuration
original_index = "uc202-rex"
chunk_index = "uc202-rex-chunks"
chunk_size = 512
chunk_overlap = 128


/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/elasticsearch/_sync/client/__init__.py:397: SecurityWarning: Connecting to 'https://noeyyalp.noe.edf.fr:29203' using TLS with verify_certs=False is insecure
  _transport = transport_class(


In [2]:
existing_docs = es.search(
    index=chunk_index,
    body={
        "size": 0,
        "aggs": {"unique_docs": {"cardinality": {"field": "original_doc_id.keyword"}}},
    }
)
print(
    f"Documents déjà traités : {existing_docs['aggregations']['unique_docs']['value']}"
)


/tmp/ipykernel_17820/367954822.py:1: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  existing_docs = es.search(
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Documents déjà traités : 7942


In [10]:
processed_ids = set()
scroll_resp = es.search(
    index=chunk_index, scroll="5m", size=1000, body={"query": {"match_all": {}}}
)
while scroll_resp["hits"]["hits"]:
    processed_ids.update(
        [doc["_source"]["original_doc_id"] for doc in scroll_resp["hits"]["hits"]]
    )
    scroll_resp = es.scroll(scroll_id=scroll_resp["_scroll_id"], scroll="5m")


/tmp/ipykernel_10068/3639086483.py:2: DeprecationWarning: The 'body' parameter is deprecated and will be removed in a future version. Instead use individual parameters.
  scroll_resp = es.search(
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'noeyyalp.noe.edf.fr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/opt/app-root/src/uc202-ipn-rex/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarnin